# Système embarqué
**Code 2 Projet Deep Learning**

Emma Vulliez, Elisa Forthoffer, Morgan Beaumé


In [ ]:
import torch
from ultralytics import YOLO

In [17]:
MODEL_PATH = "./runs/detect/train/weights/best.pt"
TAU_CONF = 0.5      # seuil confiance choisi
TAU_AREA = 0.05     # seuil aire relative choisi
CAMERA_INDEX = 0   # 0 = webcam par défaut

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device utilisé :", DEVICE)

Device utilisé : cuda


Chargement meilleur model

In [18]:
model = YOLO(MODEL_PATH)
model.to(DEVICE)

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_s

Fonction décision si atterissage safe ou non

In [19]:
def decision_landing(results, img_shape, tau_area):
    """
    Analyse les bounding boxes et décide si l'atterrissage est possible.
    """
    img_h, img_w = img_shape
    img_area = img_h * img_w

    max_relative_area = 0.0

    for box in results.boxes:
        x1, y1, x2, y2 = box.xyxy[0]
        box_area = float((x2 - x1) * (y2 - y1))
        relative_area = box_area / img_area

        if relative_area > max_relative_area:
            max_relative_area = relative_area

    safe = max_relative_area >= tau_area
    return safe, max_relative_area

In [ ]:
#test si caméra accessible
import cv2
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
print(cap.isOpened())
cap.release()

True


Utilisation caméra

In [21]:
cv2.destroyAllWindows()
cap = cv2.VideoCapture(CAMERA_INDEX)


if not cap.isOpened():
    print("Impossible d'ouvrir la caméra")
    exit()

print("Webcam activée — appuie sur 'q' pour quitter")


# Boucle principale
while True:
    ret, frame = cap.read()
    if not ret:
        print("Erreur lecture caméra")
        break

    img_h, img_w, _ = frame.shape

    # Inférence YOLO
    results = model.predict(
        source=frame,
        conf=TAU_CONF,
        device=DEVICE,
        verbose=False
    )[0]

    # Décision
    safe, rel_area = decision_landing(results, (img_h, img_w), TAU_AREA)

    # Dessiner les bounding boxes
    annotated_frame = results.plot()

    # Texte décision
    if safe:
        text = f"SAFE TO LAND | Area={rel_area:.3f}"
        color = (0, 255, 0)
    else:
        text = f"DO NOT LAND | Area={rel_area:.3f}"
        color = (0, 0, 255)

    cv2.putText(
        annotated_frame,
        text,
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        color,
        2
    )

    cv2.imshow("Landing Decision System", annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


Webcam activée — appuie sur 'q' pour quitter


In [22]:
# Nettoyage a mettre en commentaire, c'est pour fermer la caméra
cv2.destroyAllWindows()
print("Système arrêté")

Système arrêté
